# Create your own reader

ECGProcess includes established classes which reads XML and DICOM files. 
If your ECGs are stored in another format you can write your own reader 
and use it through the module's `ECGTable`.

A reader has one job: open a single file and store everything in it as a flat
dictionary (tag name to value) in the `raw_data` attribute. 
The base class, `BaseReader`, then does the rest: its `extract` method uses your configuration
file to pick values out of that dictionary, sorts them into `MetaData`,
`WaveForms`, `MedianBeats`, and `OtherData`.

To create your own reader simply inherit from `BaseReader` and make sure 
to implement a `__call__` method which explains how to read your new ECG file
format and mapping this to a raw data dictionary. 

Below we build a reader for simple CSV file where each column holds one lead.

In [1]:
import pathlib
import tempfile
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from ecgprocess.constants import CoreData as Core
from ecgprocess.errors import ReaderOutputError
from ecgprocess.example_data.examples import config_file
from ecgprocess.process import BaseReader
from ecgprocess.tabular import ECGTable
from ecgprocess.utils.config_tools import ConfigParser, DataMap
from ecgprocess.utils.general import ManagedProperty

CProc = Core.ProcessingData

## Example data

As an example we first create two small CSV files, each with two seconds of leads I and II sampled at 250 Hz. 


In [2]:
data_dir = pathlib.Path(tempfile.mkdtemp())
time = np.linspace(0, 2, 500)
paths = []
for offset, patient in enumerate(["P001", "P002"]):
    path = data_dir / f"{patient}.csv"
    pd.DataFrame(
        {
            "lead_I": np.sin(2 * np.pi * time) + offset,
            "lead_II": np.cos(2 * np.pi * time) + offset,
        }
    ).to_csv(path, index=False)
    paths.append(str(path))
pd.read_csv(paths[0]).head(3)

,lead_I,lead_II
0,0.000000,1.000000
1,0.025180,0.999683
2,0.050345,0.998732


## The reader

Next we will create the `CSVReader` reader class which can read and process these csv files. 
The new reader class includes a required `__call__` method and implements raw_data using 
the optional `set_with_setter` function, preventing the object from being overwritten. 

In [3]:
@dataclass
class CSVReader(BaseReader):
    """Open a CSV file that stores one lead in each column."""

    raw_data = ManagedProperty(CProc.RAW, dict)
    sampling_rate: int = field(default=250)

    def __call__(self, path, verbose=False, **kwargs):
        table = pd.read_csv(path)
        flat = {col: table[col].to_list() for col in table.columns}
        flat["patient"] = pathlib.Path(path).stem
        flat["rate"] = self.sampling_rate
        flat["n_samples"] = len(table)
        # raw_data stays locked, so the managed setter writes the map.
        getattr(type(self), CProc.RAW).set_with_setter(self, flat)
        return self

## Using the reader

To use the new reader we first define a configuration file and 
next use this to process the csv files.


In [4]:
# configuration file
config_path = str(data_dir / 'config.txt')
_ = config_file(path=config_path, text={
    'MetaData': [
        'unique identifier\tpatient',
        'sampling frequency (original)\trate',
        'sampling number (waveforms)\tn_samples',
    ],
    'WaveForms': ['I\tlead_I', 'II\tlead_II'],
})
parser = ConfigParser(config_path)()
parser.map(mapper=DataMap())

In [5]:
# read the files
reader = CSVReader()(paths[0]).extract(config=parser)
print(reader.MetaData['unique identifier'])
print({lead: signal.shape for lead, signal in reader.WaveForms.items()
       if signal is not None})

P001
{'I': (1000,), 'II': (1000,)}


The same reader can be handed to `ECGTable` to process several
files. These files have no median beats, so that category is turned
off with `extract_median=False`.

In [6]:
table = ECGTable(CSVReader(), path_list=paths, extract_median=False)
table = table().get_table(parsed_config=parser)
table.WaveForms.dropna().head()

,key,Sampling sequence,Lead,Voltage,Signal type
0,P001,0,I,8.881784e-19,WaveForms
1,P001,0,II,1.000000e+00,WaveForms
12,P001,1,I,9.150674e-03,WaveForms
13,P001,1,II,9.999064e-01,WaveForms
24,P001,2,I,2.518045e-02,WaveForms


## What can go wrong

`BaseReader` checks two things for you.
First, a reader without a `__call__` method cannot be created.

In [7]:
@dataclass
class NoCallReader(BaseReader):
    pass

try:
    NoCallReader()
except TypeError as error:
    print(error)

Can't instantiate abstract class NoCallReader with abstract method __call__



Second, `ECGTable` checks that the reader creates every data object requested. 
This mostly matters if you override `extract`: always call
`super().extract(...)` from your version, or set all the categories yourself.

As an example the `ForgetfulReader` reader below does not define the waveform data object
while the `ECGTable` call does expect this to be available. 



In [8]:
@dataclass
class ForgetfulReader(CSVReader):
    def extract(self, config, **kwargs):
        self.MetaData = {'unique identifier': self.raw_data['patient']}
        return self

try:
    ECGTable(ForgetfulReader(), path_list=paths,
             extract_median=False)().get_table(parsed_config=parser)
except ReaderOutputError as error:
    print(error)

`ForgetfulReader` did not set `WaveForms` after `extract`. Either populate it in the reader, or set the matching extract_meta/extract_wave/extract_median flag to `False` on ECGTable if the reader does not provide this data.
